# Seq Len Hidden State Debug

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from pfns.experiments.model_benchmarks.model_registry import get_models_from_names
from pfns.experiments.model_benchmarks.seq_len_debug_utils import (
    auto_select_position_metric_layers,
    plot_avg_metric_per_layer_per_head,
    plot_position_metric_per_layer,
    plot_recurrent_metric_per_head,
    run_position_metric_tracking,
    run_hidden_state_tracking,
    summarize_hidden_state_by_seqlen,
)
from pfns.utils import get_default_device
from pathlib import Path

plt.rcParams['figure.dpi'] = 400


## Config


In [ ]:
EXPERIMENT = {
    'name': 'seq_len_hidden_state_debug',
    'num_classes': 5,
    'num_features': 10,
    'num_test_samples': 100,
    'num_repetitions': 10,
    'data_generation_seed': 42,
    'seqlen_list': [250, 500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000],
}

MODEL_NAMES = [
    'Linear_Attention_Non_Causal',
    'Linear_Attention_Causal_Comb_ST',
    #'Oracle_Hidden_State_Linear_Attention_Non_Causal_base'
    # 'GLA_Comb_ST'
]

TRAINING_CONTEXT_LENGTH = 1_000

# Toggle between the original per-run traces and seq-len violin distributions.
HIDDEN_STATE_PLOT_MODE = 'violin'  # 'violin' or 'individual_runs'
HIDDEN_STATE_DISTRIBUTION_ALPHA = 0.3
HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC = 0.4

HIDDEN_STATE_CACHE_DIR = Path("hidden_state_cache")
HIDDEN_STATE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN_STATE_CACHE_OVERWRITE = False

device = str(get_default_device())
models_to_compare = get_models_from_names(MODEL_NAMES)

print(f'Device: {device}')
print(f'Models: {list(models_to_compare)}')
print(f'Repetitions: {EXPERIMENT["num_repetitions"]}')
print(f'Seq lens: {EXPERIMENT["seqlen_list"]}')
print(f'Hidden-state plot mode: {HIDDEN_STATE_PLOT_MODE}')

## Run Hidden-State Tracking


In [ ]:
from pathlib import Path

import pandas as pd

from pfns.experiments.model_benchmarks.hashing import experiment_payload_hash
from pfns.experiments.model_benchmarks.model_registry import functional_model_config

HIDDEN_STATE_CACHE_DIR = Path("hidden_state_cache")
HIDDEN_STATE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN_STATE_CACHE_OVERWRITE = False

hidden_state_cache_key = experiment_payload_hash(
    experiment_payload={
        "experiment": EXPERIMENT,
        "models_to_compare": {
            model_name: functional_model_config(model_config)
            for model_name, model_config in models_to_compare.items()
        },
    }
)
hidden_state_cache_path = (
    HIDDEN_STATE_CACHE_DIR / f"{EXPERIMENT['name']}_{hidden_state_cache_key}.pkl"
)

if hidden_state_cache_path.exists() and not HIDDEN_STATE_CACHE_OVERWRITE:
    hidden_state_df = pd.read_pickle(hidden_state_cache_path)
    print(f"Loaded hidden_state_df from cache: {hidden_state_cache_path}")
else:
    hidden_state_df = run_hidden_state_tracking(
        experiment=EXPERIMENT,
        models_to_compare=models_to_compare,
        device=device,
    )
    hidden_state_df.to_pickle(hidden_state_cache_path)
    print(f"Saved hidden_state_df to cache: {hidden_state_cache_path}")

print("hidden_state_df rows:", len(hidden_state_df))
hidden_state_df.head()

hidden_state_df_plot = hidden_state_df
print('hidden_state_df_plot rows:', len(hidden_state_df_plot))
hidden_state_df_plot.head()


## Summaries and Plots


In [ ]:
summary_df = summarize_hidden_state_by_seqlen(hidden_state_df)

print('summary_df rows:', len(summary_df))
summary_df.head()

### Metric Guide
- `abs_max_mean`: average largest absolute entry in each cached `kv_state` head matrix.
- `frobenius_norm_mean`: per-head Frobenius norm of the cached `kv_state` matrices.
- `spectral_norm_mean`: per-head spectral norm of the cached `kv_state` matrices.
- `effective_rank_mean`: effective rank `exp(H(sigma / ||sigma||_1))` of each cached `kv_state` matrix.
- `stable_rank_mean`: stable rank `||KV||_F^2 / ||KV||_2^2` of each cached `kv_state` matrix.

In [ ]:
all_tensors = sorted(summary_df['tensor_name'].astype(str).unique().tolist())
print(f'Recurrent-state tensors available: {len(all_tensors)}')
for name in all_tensors:
    print(' ', name)

In [ ]:
shared_plot_kwargs = {
    'plot_mode': HIDDEN_STATE_PLOT_MODE,
    'distribution_alpha': HIDDEN_STATE_DISTRIBUTION_ALPHA,
    'distribution_width_frac': HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC,
    #'log_y': True,
}

generic_plot_specs = [
    #('abs_max', plot_recurrent_metric_per_head, 'Final hidden-state abs max vs sequence length'),
    ('frobenius_norm', plot_recurrent_metric_per_head, 'Final hidden-state Frobenius norm vs sequence length'),
    #('spectral_norm', plot_recurrent_metric_per_head, 'Final hidden-state spectral norm vs sequence length'),
    #('effective_rank', plot_recurrent_metric_per_layer, 'Final hidden-state effective rank vs sequence length'),
    #('stable_rank', plot_recurrent_metric_per_layer, 'Final hidden-state stable rank vs sequence length'),
]

linear_attention_plot_specs = [
    ('k_sum_norm', plot_recurrent_metric_per_head, 'Final K-sum norm vs sequence length'),
    ('joint_hidden_state_norm', plot_recurrent_metric_per_head, 'Final joint hidden-state norm vs sequence length'),
    ('kv_over_ksum_ratio', plot_recurrent_metric_per_head, 'Final KV-over-K-sum ratio vs sequence length'),
]

for section_name, plot_specs in [
    ('Generic final-state plots', generic_plot_specs),
    ('Linear-attention-specific final-state plots', linear_attention_plot_specs),
]:
    print(section_name)
    any_plotted = False
    for metric, plot_fn, title_prefix in plot_specs:
        if metric not in hidden_state_df_plot.columns:
            continue
        metric_values = pd.to_numeric(hidden_state_df_plot[metric], errors='coerce')
        if not metric_values.notna().any():
            continue
        plot_fn(
            hidden_state_df_plot,
            metric=metric,
            title_prefix=title_prefix,
            training_context_length=TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        )
        any_plotted = True
    if not any_plotted:
        print(f'- No finite metrics available for: {section_name}')


print('Per-head metric summaries across sequence lengths:')
for metric, title_prefix in [
    ('effective_rank', 'Effective rank per layer'),
    ('stable_rank', 'Stable rank per layer'),
]:
    if metric not in hidden_state_df_plot.columns:
        continue
    metric_values = pd.to_numeric(hidden_state_df_plot[metric], errors='coerce')
    if not metric_values.notna().any():
        continue
    plot_avg_metric_per_layer_per_head(
        hidden_state_df_plot,
        metric=metric,
        title_prefix=title_prefix,
    )


## State Metrics vs Position


In [ ]:
from pfns.experiments.model_benchmarks.hashing import experiment_payload_hash
from pfns.experiments.model_benchmarks.model_registry import functional_model_config

POSITION_METRIC_EXPERIMENT = {
    'name': 'seq_len_qk_state_position_debug_with_output_norm_svd_alignment_v4',
    'model_names': [
        'Linear_Attention_Causal_Comb_ST',
        'Linear_Attention_Non_Causal',
    ],
    'seqlen': 32_000,
    'num_repetitions': 1,
    'data_generation_seed': EXPERIMENT['data_generation_seed'],
    'num_classes': EXPERIMENT['num_classes'],
    'num_features': EXPERIMENT['num_features'],
    'num_test_samples': EXPERIMENT['num_test_samples'],
    'log_x': True,
}

POSITION_METRIC_LAYER_SELECTION = list(range(15))  # None -> auto-pick first / middle / last
POSITION_METRIC_CACHE_OVERWRITE = False

position_metric_model_configs = get_models_from_names(POSITION_METRIC_EXPERIMENT['model_names'])
position_metric_cache_key = experiment_payload_hash(
    experiment_payload={
        'experiment': POSITION_METRIC_EXPERIMENT,
        'models_to_compare': {
            model_name: functional_model_config(model_config)
            for model_name, model_config in position_metric_model_configs.items()
        },
    }
)
position_metric_cache_path = (
    HIDDEN_STATE_CACHE_DIR / f"{POSITION_METRIC_EXPERIMENT['name']}_{position_metric_cache_key}.pkl"
)
position_metric_partial_cache_path = position_metric_cache_path.with_name(
    f'{position_metric_cache_path.stem}_partial.pkl'
)

if position_metric_cache_path.exists() and not POSITION_METRIC_CACHE_OVERWRITE:
    position_metric_df = pd.read_pickle(position_metric_cache_path)
    print(f'Loaded position_metric_df from cache: {position_metric_cache_path}')
else:
    position_metric_df = run_position_metric_tracking(
        experiment=POSITION_METRIC_EXPERIMENT,
        device=device,
        training_context_length=TRAINING_CONTEXT_LENGTH,
        partial_cache_path=position_metric_partial_cache_path,
    )
    position_metric_df.to_pickle(position_metric_cache_path)
    if position_metric_partial_cache_path.exists():
        position_metric_partial_cache_path.unlink()
    print(f'Saved position_metric_df to cache: {position_metric_cache_path}')

print('position_metric_df rows:', len(position_metric_df))
position_metric_df.head()

In [ ]:
available_layers = sorted(position_metric_df['layer_idx'].astype(int).unique().tolist())
selected_layers = (
    POSITION_METRIC_LAYER_SELECTION
    if POSITION_METRIC_LAYER_SELECTION is not None
    else auto_select_position_metric_layers(available_layers)
)
print('Selected layers:', selected_layers)

metric_specs = [
    ('k_sum_norm', '||k_sum||'),
    ('kv_state_norm', '||kv_state||'),
    ('kv_over_ksum_ratio', '||kv_state|| / ||k_sum||'),
    ('output_norm', '||(q^T KV) / (q^T k_sum + eps)||'),
]

SHOW_STD_SHADING = False
model_order = POSITION_METRIC_EXPERIMENT['model_names']
for metric_col, metric_label in metric_specs:
    plot_position_metric_per_layer(
        position_metric_df,
        metric=metric_col,
        title_prefix=f'{metric_label} vs token position',
        model_order=model_order,
        layer_indices=selected_layers,
        log_x=bool(POSITION_METRIC_EXPERIMENT['log_x']),
        y_scale='log',
        show_std_shading=SHOW_STD_SHADING,
        seqlen=int(POSITION_METRIC_EXPERIMENT['seqlen']),
    )
